# Your first predictive model

Instructor preparation: 60–90 minutes. Predict a fictional delivery time from route distance and stop count known before dispatch. The independent synthetic observations support a random split for this exercise; real repeated-customer or temporal data may need grouped or chronological splits. The simple data-generating formula makes this deliberately easy; it is not evidence of real-world accuracy.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
rng = np.random.default_rng(42)
X = pd.DataFrame({'distance_km': rng.uniform(1, 30, 240), 'stops': rng.integers(1, 8, 240)})
y = 12 + 2.4 * X['distance_km'] + 4 * X['stops'] + rng.normal(0, 6, len(X))
print(X.head())
print('Target: delivery duration in minutes')


## Split before fitting

Keep a quarter of the observations unseen during fitting. The fixed seed makes this demonstration repeatable. Do not tune against these test scores. For model selection, use cross-validation on the training set and retain an untouched final test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print('Training rows:', len(X_train), 'Test rows:', len(X_test))


## Compare a baseline and one preselected model

The baseline always predicts the training mean. Linear regression estimates coefficients from training examples. `fit` learns parameters; `predict` applies them.

In [ ]:
baseline = DummyRegressor(strategy='mean').fit(X_train, y_train)
model = LinearRegression().fit(X_train, y_train)
baseline_mae = mean_absolute_error(y_test, baseline.predict(X_test))
model_mae = mean_absolute_error(y_test, model.predict(X_test))
print(f'Baseline test MAE: {baseline_mae:.2f} minutes')
print(f'Linear model test MAE: {model_mae:.2f} minutes')
print('Learned coefficients:', dict(zip(X.columns, model.coef_)))
print(f'Intercept: {model.intercept_:.2f} minutes')


## Interpret error

MAE is the average absolute error on the held-out examples. It is not a guarantee that each future prediction lies within that error. The model is useful in this demonstration only if it improves on the baseline; operational usefulness also depends on costs, uncertainty and representativeness.

In [ ]:
new_route = pd.DataFrame({'distance_km':[10.0], 'stops':[3]})
print(f'Predicted duration: {model.predict(new_route)[0]:.1f} minutes')
print(pd.DataFrame({'actual':y_test, 'predicted':model.predict(X_test)}).head().round(1))


## Teach-back questions

1. What is one row? One route.
2. What is X? Distance and planned stops. What is y? Observed delivery duration.
3. What does fit do? Estimates model parameters from training data.
4. Why a baseline? To check whether the model improves on a simple reference.
5. Could we use actual arrival time as an input? No: it would not be available before dispatch and would leak the outcome.
6. Are coefficients causal effects? No; fitted associations alone do not establish causation.
7. Why not keep improving the model using this test set? Repeated choices based on it undermine its role as an unbiased final check.

Optional practice: change the noise level and rebuild the experiment. Treat this as a synthetic simulation, not further optimisation against the original test set.